In [ ]:
import random
import threading
import time

import pyarrow as pa
import pyarrow.compute as pc
import ursid as us

s = us.local()
book = s.input_table(
    {"id": us.int64(), "sym": us.string(), "price": us.float64(), "size": us.int64()},
    key="id",
)
syms = ["AAPL", "MSFT", "NVDA", "TSLA", "GOOG", "AMZN", "META", "JPM"]
rng = random.Random(1)


def feed():
    while True:
        n = 2_000
        book.upsert(
            pa.table(
                {
                    "id": pa.array(
                        [rng.randrange(100_000) for _ in range(n)], pa.int64()
                    ),
                    "sym": [rng.choice(syms) for _ in range(n)],
                    "price": [round(rng.gauss(100, 10), 2) for _ in range(n)],
                    "size": pa.array(
                        [rng.randrange(1, 1000) for _ in range(n)], pa.int64()
                    ),
                }
            )
        )
        time.sleep(0.1)


threading.Thread(target=feed, daemon=True).start()
time.sleep(0.5)

In [ ]:
book

In [ ]:
n = 1_000_000
big = s.static_table(
    pa.table(
        {
            "i": pa.array(range(n), pa.int64()),
            "x": pc.random(n),
            "sym": [rng.choice(syms) for _ in range(n)],
        }
    )
)
big

In [ ]:
book.sort("price desc")